<a href="https://colab.research.google.com/github/DEEPLERZERA/Iniciacao-Cientifica-2026-predicao-de-parametros-climaticos/blob/main/Barueri_Sao_Paulo_2025_modelo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Teste com variáveis selecionadas - Regressão Linear (OLS) e Random Forest

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando os Dados de Barueri
nome_arquivo = 'Barueri_Sao_Paulo_2025.xlsx'
df = pd.read_excel(nome_arquivo)

# Tratamento de segurança para os separadores decimais
for col in df.columns:
    if df[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df[col] = pd.to_numeric(df[col].str.replace(',', '.'), errors='coerce')
        except:
            df[col] = pd.to_numeric(df[col], errors='coerce')

target_col = 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'
X_cols = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)'
]

df_model = df[[target_col] + X_cols].copy()

# 2. Tratamento Avançado de Dados Faltantes (Data Quality Fix)
# Garante que não perderemos os primeiros meses do ano por conta de sensores desativados
df_model['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'] = df_model['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'].fillna(0)
df_model['RADIACAO GLOBAL (Kj/m²)'] = df_model['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_model = df_model.interpolate(method='linear', limit_direction='both').dropna()

X = df_model[X_cols]
y = df_model[target_col]

# =====================================================================
# ETAPA 1: DIVISÃO CRONOLÓGICA (80% Passado para Treino / 20% Futuro para Teste)
# =====================================================================
ponto_de_corte = int(len(df_model) * 0.8)

X_train = X.iloc[:ponto_de_corte]
y_train = y.iloc[:ponto_de_corte]

X_test = X.iloc[ponto_de_corte:]
y_test = y.iloc[ponto_de_corte:]

# =====================================================================
# ETAPA 2: AVALIAÇÃO ESTATÍSTICA OLS (Baseada no Treino)
# =====================================================================
print("--- RESULTADOS ESTATÍSTICOS (OLS) - BARUERI C1 ---")
X_train_sm = sm.add_constant(X_train)
model_ols = sm.OLS(y_train, X_train_sm).fit()
print(model_ols.summary())

# =====================================================================
# ETAPA 3: AVALIAÇÃO PREDITIVA RANDOM FOREST (Baseada no Teste)
# =====================================================================
print("\n--- RESULTADOS RANDOM FOREST - BARUERI C1 ---")
rf = RandomForestRegressor(random_state=42, n_estimators=100, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
r2_rf = r2_score(y_test, y_pred)
mae_rf = mean_absolute_error(y_test, y_pred)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"R² do Random Forest em dados futuros: {r2_rf:.4f}")
print(f"MAE do Random Forest: {mae_rf:.4f} °C")
print(f"RMSE do Random Forest: {rmse_rf:.4f} °C")

importances = pd.Series(rf.feature_importances_, index=X_cols).sort_values(ascending=False)
print("\nImportância das Variáveis no Random Forest (em % de contribuição):")
print(importances * 100)

--- RESULTADOS ESTATÍSTICOS (OLS) - BARUERI C1 ---
                                         OLS Regression Results                                         
Dep. Variable:     TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)   R-squared:                       0.766
Model:                                                      OLS   Adj. R-squared:                  0.766
Method:                                           Least Squares   F-statistic:                     3827.
Date:                                          Sat, 27 Jun 2026   Prob (F-statistic):               0.00
Time:                                                  17:20:18   Log-Likelihood:                -16021.
No. Observations:                                          7008   AIC:                         3.206e+04
Df Residuals:                                              7001   BIC:                         3.210e+04
Df Model:                                                     6                                         
Cova

## Teste com todas as variáveis (Sem filtro) - Regressão Linear (OLS) e Random Forest

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando os Dados de Barueri
nome_arquivo = 'Barueri_Sao_Paulo_2025.xlsx'
df = pd.read_excel(nome_arquivo)

# Tratamento de segurança para os separadores decimais
for col in df.columns:
    if df[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df[col] = pd.to_numeric(df[col].str.replace(',', '.'), errors='coerce')
        except:
            df[col] = pd.to_numeric(df[col], errors='coerce')

target_col = 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'

# Variáveis do Cenário 2 (Todas as 11 físicas, sem tempo)
X_cols_2 = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)',
    'PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)',
    'UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)',
    'VENTO, RAJADA MAXIMA (m/s)'
]

df_model_2 = df[[target_col] + X_cols_2].copy()

# 2. Tratamento Avançado de Dados Faltantes (Data Quality Fix)
df_model_2['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'] = df_model_2['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'].fillna(0)
df_model_2['RADIACAO GLOBAL (Kj/m²)'] = df_model_2['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_model_2 = df_model_2.interpolate(method='linear', limit_direction='both').dropna()

X_2 = df_model_2[X_cols_2]
y_2 = df_model_2[target_col]

# =====================================================================
# ETAPA 1: DIVISÃO CRONOLÓGICA (80% Passado para Treino / 20% Futuro para Teste)
# =====================================================================
ponto_de_corte = int(len(df_model_2) * 0.8)

X_train_2 = X_2.iloc[:ponto_de_corte]
y_train_2 = y_2.iloc[:ponto_de_corte]

X_test_2 = X_2.iloc[ponto_de_corte:]
y_test_2 = y_2.iloc[ponto_de_corte:]

# =====================================================================
# ETAPA 2: AVALIAÇÃO ESTATÍSTICA OLS (Baseada no Treino)
# =====================================================================
print("--- RESULTADOS ESTATÍSTICOS (OLS) - BARUERI C2 ---")
X_train_sm_2 = sm.add_constant(X_train_2)
model_ols_2 = sm.OLS(y_train_2, X_train_sm_2).fit()
print(model_ols_2.summary())

# =====================================================================
# ETAPA 3: AVALIAÇÃO PREDITIVA RANDOM FOREST (Baseada no Teste)
# =====================================================================
print("\n--- RESULTADOS RANDOM FOREST - BARUERI C2 ---")
rf_2 = RandomForestRegressor(random_state=42, n_estimators=100, n_jobs=-1)
rf_2.fit(X_train_2, y_train_2)

y_pred_2 = rf_2.predict(X_test_2)
r2_rf_2 = r2_score(y_test_2, y_pred_2)
mae_rf_2 = mean_absolute_error(y_test_2, y_pred_2)
rmse_rf_2 = np.sqrt(mean_squared_error(y_test_2, y_pred_2))

print(f"R² do Random Forest em dados futuros: {r2_rf_2:.4f}")
print(f"MAE do Random Forest: {mae_rf_2:.4f} °C")
print(f"RMSE do Random Forest: {rmse_rf_2:.4f} °C")

importances_2 = pd.Series(rf_2.feature_importances_, index=X_cols_2).sort_values(ascending=False)
print("\nImportância das Variáveis no Random Forest (em % de contribuição):")
print(importances_2 * 100)

--- RESULTADOS ESTATÍSTICOS (OLS) - BARUERI C2 ---
                                         OLS Regression Results                                         
Dep. Variable:     TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)   R-squared:                       0.772
Model:                                                      OLS   Adj. R-squared:                  0.772
Method:                                           Least Squares   F-statistic:                     2158.
Date:                                          Sat, 27 Jun 2026   Prob (F-statistic):               0.00
Time:                                                  19:11:54   Log-Likelihood:                -15929.
No. Observations:                                          7008   AIC:                         3.188e+04
Df Residuals:                                              6996   BIC:                         3.196e+04
Df Model:                                                    11                                         
Cova

## Teste com variáveis selecionadas + Hora e mês - Regressão Linear (OLS) e Random Forest

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando os Dados de Barueri
nome_arquivo = 'Barueri_Sao_Paulo_2025.xlsx'
df = pd.read_excel(nome_arquivo)

# Tratamento de segurança para os separadores decimais
for col in df.columns:
    if df[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df[col] = pd.to_numeric(df[col].str.replace(',', '.'), errors='coerce')
        except:
            df[col] = pd.to_numeric(df[col], errors='coerce')

target_col = 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'

# Engenharia de Atributos para o Cenário 3 (Extraindo componentes de tempo)
df['Hora'] = df['Hora UTC'].str.replace(' UTC', '').astype(int) / 100
df['Mes'] = pd.to_datetime(df['Data']).dt.month

# Variáveis do Cenário 3 (Físicas Selecionadas + Tempo)
X_cols_3 = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)',
    'Hora',
    'Mes'
]

df_model_3 = df[[target_col] + X_cols_3].copy()

# 2. Tratamento Consistente de Dados Faltantes
df_model_3['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'] = df_model_3['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'].fillna(0)
df_model_3['RADIACAO GLOBAL (Kj/m²)'] = df_model_3['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_model_3 = df_model_3.interpolate(method='linear', limit_direction='both').dropna()

X_3 = df_model_3[X_cols_3]
y_3 = df_model_3[target_col]

# =====================================================================
# ETAPA 1: DIVISÃO CRONOLÓGICA (80% Treino / 20% Teste)
# =====================================================================
ponto_de_corte = int(len(df_model_3) * 0.8)

X_train_3 = X_3.iloc[:ponto_de_corte]
y_train_3 = y_3.iloc[:ponto_de_corte]

X_test_3 = X_3.iloc[ponto_de_corte:]
y_test_3 = y_3.iloc[ponto_de_corte:]

# =====================================================================
# ETAPA 2: AVALIAÇÃO ESTATÍSTICA OLS (Baseada no Treino)
# =====================================================================
print("--- RESULTADOS ESTATÍSTICOS (OLS) - BARUERI C3 ---")
X_train_sm_3 = sm.add_constant(X_train_3)
model_ols_3 = sm.OLS(y_train_3, X_train_sm_3).fit()
print(model_ols_3.summary())

# =====================================================================
# ETAPA 3: AVALIAÇÃO PREDITIVA RANDOM FOREST (Baseada no Teste)
# =====================================================================
print("\n--- RESULTADOS RANDOM FOREST - BARUERI C3 ---")
rf_3 = RandomForestRegressor(random_state=42, n_estimators=100, n_jobs=-1)
rf_3.fit(X_train_3, y_train_3)

y_pred_3 = rf_3.predict(X_test_3)
r2_rf_3 = r2_score(y_test_3, y_pred_3)
mae_rf_3 = mean_absolute_error(y_test_3, y_pred_3)
rmse_rf_3 = np.sqrt(mean_squared_error(y_test_3, y_pred_3))

print(f"R² do Random Forest em dados futuros: {r2_rf_3:.4f}")
print(f"MAE do Random Forest: {mae_rf_3:.4f} °C")
print(f"RMSE do Random Forest: {rmse_rf_3:.4f} °C")

importances_3 = pd.Series(rf_3.feature_importances_, index=X_cols_3).sort_values(ascending=False)
print("\nImportância das Variáveis no Random Forest (em % de contribuição):")
print(importances_3 * 100)

--- RESULTADOS ESTATÍSTICOS (OLS) - BARUERI C3 ---
                                         OLS Regression Results                                         
Dep. Variable:     TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)   R-squared:                       0.786
Model:                                                      OLS   Adj. R-squared:                  0.786
Method:                                           Least Squares   F-statistic:                     3221.
Date:                                          Sat, 27 Jun 2026   Prob (F-statistic):               0.00
Time:                                                  19:27:39   Log-Likelihood:                -15707.
No. Observations:                                          7008   AIC:                         3.143e+04
Df Residuals:                                              6999   BIC:                         3.149e+04
Df Model:                                                     8                                         
Cova

## Teste com todas as variáveis (Sem filtro) + Hora e mês - Regressão Linear (OLS) e Random Forest

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando os Dados de Barueri
nome_arquivo = 'Barueri_Sao_Paulo_2025.xlsx'
df = pd.read_excel(nome_arquivo)

# Tratamento de segurança para os separadores decimais
for col in df.columns:
    if df[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df[col] = pd.to_numeric(df[col].str.replace(',', '.'), errors='coerce')
        except:
            df[col] = pd.to_numeric(df[col], errors='coerce')

target_col = 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'

# Engenharia de Atributos para o Cenário 4 (Extraindo tempo)
df['Hora'] = df['Hora UTC'].str.replace(' UTC', '').astype(int) / 100
df['Mes'] = pd.to_datetime(df['Data']).dt.month

# Variáveis do Cenário 4 (Todas as 11 físicas + Tempo)
X_cols_4 = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)',
    'PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)',
    'UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)',
    'VENTO, RAJADA MAXIMA (m/s)',
    'Hora',
    'Mes'
]

df_model_4 = df[[target_col] + X_cols_4].copy()

# 2. Tratamento Consistente de Dados Faltantes
df_model_4['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'] = df_model_4['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'].fillna(0)
df_model_4['RADIACAO GLOBAL (Kj/m²)'] = df_model_4['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_model_4 = df_model_4.interpolate(method='linear', limit_direction='both').dropna()

X_4 = df_model_4[X_cols_4]
y_4 = df_model_4[target_col]

# =====================================================================
# ETAPA 1: DIVISÃO CRONOLÓGICA (80% Treino / 20% Teste)
# =====================================================================
ponto_de_corte = int(len(df_model_4) * 0.8)

X_train_4 = X_4.iloc[:ponto_de_corte]
y_train_4 = y_4.iloc[:ponto_de_corte]

X_test_4 = X_4.iloc[ponto_de_corte:]
y_test_4 = y_4.iloc[ponto_de_corte:]

# =====================================================================
# ETAPA 2: AVALIAÇÃO ESTATÍSTICA OLS (Baseada no Treino)
# =====================================================================
print("--- RESULTADOS ESTATÍSTICOS (OLS) - BARUERI C4 ---")
X_train_sm_4 = sm.add_constant(X_train_4)
model_ols_4 = sm.OLS(y_train_4, X_train_sm_4).fit()
print(model_ols_4.summary())

# =====================================================================
# ETAPA 3: AVALIAÇÃO PREDITIVA RANDOM FOREST (Baseada no Teste)
# =====================================================================
print("\n--- RESULTADOS RANDOM FOREST - BARUERI C4 ---")
rf_4 = RandomForestRegressor(random_state=42, n_estimators=100, n_jobs=-1)
rf_4.fit(X_train_4, y_train_4)

y_pred_4 = rf_4.predict(X_test_4)
r2_rf_4 = r2_score(y_test_4, y_pred_4)
mae_rf_4 = mean_absolute_error(y_test_4, y_pred_4)
rmse_rf_4 = np.sqrt(mean_squared_error(y_test_4, y_pred_4))

print(f"R² do Random Forest em dados futuros: {r2_rf_4:.4f}")
print(f"MAE do Random Forest: {mae_rf_4:.4f} °C")
print(f"RMSE do Random Forest: {rmse_rf_4:.4f} °C")

importances_4 = pd.Series(rf_4.feature_importances_, index=X_cols_4).sort_values(ascending=False)
print("\nImportância das Variáveis no Random Forest (em % de contribuição):")
print(importances_4 * 100)

--- RESULTADOS ESTATÍSTICOS (OLS) - BARUERI C4 ---
                                         OLS Regression Results                                         
Dep. Variable:     TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)   R-squared:                       0.791
Model:                                                      OLS   Adj. R-squared:                  0.790
Method:                                           Least Squares   F-statistic:                     2034.
Date:                                          Sat, 27 Jun 2026   Prob (F-statistic):               0.00
Time:                                                  19:52:05   Log-Likelihood:                -15634.
No. Observations:                                          7008   AIC:                         3.130e+04
Df Residuals:                                              6994   BIC:                         3.139e+04
Df Model:                                                    13                                         
Cova

## Teste com variáveis selecionadas - Gradient Boosting

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando os Dados de Barueri
nome_arquivo = 'Barueri_Sao_Paulo_2025.xlsx'
df = pd.read_excel(nome_arquivo)

# Limpeza de segurança
for col in df.columns:
    if df[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df[col] = pd.to_numeric(df[col].str.replace(',', '.'), errors='coerce')
        except:
            df[col] = pd.to_numeric(df[col], errors='coerce')

target_col = 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'

# 2. Variáveis Selecionadas (O cenário puro, sem Mês e Hora)
X_cols = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)'
]

df_model = df[[target_col] + X_cols].copy()

# Tratamento específico para as lacunas dos sensores de Barueri
df_model['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'] = df_model['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'].fillna(0)
df_model['RADIACAO GLOBAL (Kj/m²)'] = df_model['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_model = df_model.interpolate(method='linear', limit_direction='both').dropna()

X = df_model[X_cols]
y = df_model[target_col]

# 3. Divisão Cronológica (80% Treino / 20% Teste)
ponto_de_corte = int(len(df_model) * 0.8)
X_train, X_test = X.iloc[:ponto_de_corte], X.iloc[ponto_de_corte:]
y_train, y_test = y.iloc[:ponto_de_corte], y.iloc[ponto_de_corte:]

# =====================================================================
# MACHINE LEARNING: GRADIENT BOOSTING
# =====================================================================
print("=======================================================")
print("MODELO: GRADIENT BOOSTING (CENÁRIO 1) - BARUERI")
print("=======================================================\n")

# n_estimators=100 (100 árvores sequenciais)
# learning_rate=0.1 (tamanho do "passo" que ele dá para consertar o erro)
gb_model = GradientBoostingRegressor(random_state=42, n_estimators=100, learning_rate=0.1)
gb_model.fit(X_train, y_train)

y_pred = gb_model.predict(X_test)
r2_gb = r2_score(y_test, y_pred)

print(f"R² do Gradient Boosting em dados futuros: {r2_gb:.4f}")

importances_gb = pd.Series(gb_model.feature_importances_, index=X_cols).sort_values(ascending=False)
print("\nImportância das Variáveis (em %):")
print(importances_gb * 100)

MODELO: GRADIENT BOOSTING (CENÁRIO 1) - BARUERI

R² do Gradient Boosting em dados futuros: 0.7733

Importância das Variáveis (em %):
UMIDADE RELATIVA DO AR, HORARIA (%)                      46.893776
PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)    22.878916
RADIACAO GLOBAL (Kj/m²)                                  16.987239
VENTO, VELOCIDADE HORARIA (m/s)                          11.860456
VENTO, DIREÇÃO HORARIA (gr) (° (gr))                      1.368974
PRECIPITAÇÃO TOTAL, HORÁRIO (mm)                          0.010639
dtype: float64


## Teste com todas as variáveis (Sem filtro) - Gradient Boosting

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando os Dados de Barueri
nome_arquivo = 'Barueri_Sao_Paulo_2025.xlsx'
df = pd.read_excel(nome_arquivo)

# Limpeza de segurança
for col in df.columns:
    if df[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df[col] = pd.to_numeric(df[col].str.replace(',', '.'), errors='coerce')
        except:
            df[col] = pd.to_numeric(df[col], errors='coerce')

target_col = 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'

# 2. Variáveis Selecionadas (Todas as 11 variáveis físicas, sem Mês e Hora)
X_cols = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)',
    'PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)',
    'UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)',
    'VENTO, RAJADA MAXIMA (m/s)'
]

df_model = df[[target_col] + X_cols].copy()

# Tratamento específico para as lacunas dos sensores de Barueri
df_model['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'] = df_model['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'].fillna(0)
df_model['RADIACAO GLOBAL (Kj/m²)'] = df_model['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_model = df_model.interpolate(method='linear', limit_direction='both').dropna()

X = df_model[X_cols]
y = df_model[target_col]

# 3. Divisão Cronológica (80% Treino / 20% Teste)
ponto_de_corte = int(len(df_model) * 0.8)
X_train, X_test = X.iloc[:ponto_de_corte], X.iloc[ponto_de_corte:]
y_train, y_test = y.iloc[:ponto_de_corte], y.iloc[ponto_de_corte:]

# =====================================================================
# MACHINE LEARNING: GRADIENT BOOSTING
# =====================================================================
print("=======================================================")
print("MODELO: GRADIENT BOOSTING (CENÁRIO 2) - BARUERI")
print("=======================================================\n")

# n_estimators=100 (100 árvores sequenciais)
# learning_rate=0.1 (tamanho do "passo" que ele dá para consertar o erro)
gb_model = GradientBoostingRegressor(random_state=42, n_estimators=100, learning_rate=0.1)
gb_model.fit(X_train, y_train)

y_pred = gb_model.predict(X_test)
r2_gb = r2_score(y_test, y_pred)

print(f"R² do Gradient Boosting em dados futuros: {r2_gb:.4f}")

importances_gb = pd.Series(gb_model.feature_importances_, index=X_cols).sort_values(ascending=False)
print("\nImportância das Variáveis (em %):")
print(importances_gb * 100)

MODELO: GRADIENT BOOSTING (CENÁRIO 2) - BARUERI

R² do Gradient Boosting em dados futuros: 0.7771

Importância das Variáveis (em %):
UMIDADE RELATIVA DO AR, HORARIA (%)                      25.049623
UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)                 19.374937
RADIACAO GLOBAL (Kj/m²)                                  16.123229
PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)         11.434120
VENTO, VELOCIDADE HORARIA (m/s)                          11.175013
PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)     8.537086
UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)                  2.693633
VENTO, RAJADA MAXIMA (m/s)                                2.682907
PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)           1.688929
VENTO, DIREÇÃO HORARIA (gr) (° (gr))                      1.239519
PRECIPITAÇÃO TOTAL, HORÁRIO (mm)                          0.001004
dtype: float64


## Teste com variáveis selecionadas + Hora e mês - Gradient Boosting

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando os Dados de Barueri
nome_arquivo = 'Barueri_Sao_Paulo_2025.xlsx'
df = pd.read_excel(nome_arquivo)

# Limpeza de segurança
for col in df.columns:
    if df[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df[col] = pd.to_numeric(df[col].str.replace(',', '.'), errors='coerce')
        except:
            df[col] = pd.to_numeric(df[col], errors='coerce')

target_col = 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'

# 2. Engenharia de Atributos (Extraindo o Tempo)
df['Hora'] = df['Hora UTC'].str.replace(' UTC', '').astype(int) / 100
df['Mes'] = pd.to_datetime(df['Data']).dt.month

# 3. Variáveis Selecionadas (Física Limpa + Tempo)
X_cols = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)',
    'Hora',
    'Mes'
]

df_model = df[[target_col] + X_cols].copy()

# Tratamento específico para as lacunas dos sensores de Barueri
df_model['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'] = df_model['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'].fillna(0)
df_model['RADIACAO GLOBAL (Kj/m²)'] = df_model['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_model = df_model.interpolate(method='linear', limit_direction='both').dropna()

X = df_model[X_cols]
y = df_model[target_col]

# 4. Divisão Cronológica (80% Treino / 20% Teste)
ponto_de_corte = int(len(df_model) * 0.8)
X_train, X_test = X.iloc[:ponto_de_corte], X.iloc[ponto_de_corte:]
y_train, y_test = y.iloc[:ponto_de_corte], y.iloc[ponto_de_corte:]

# =====================================================================
# MACHINE LEARNING: GRADIENT BOOSTING
# =====================================================================
print("=======================================================")
print("MODELO: GRADIENT BOOSTING (CENÁRIO 3) - BARUERI")
print("=======================================================\n")

# n_estimators=100 (100 árvores sequenciais)
# learning_rate=0.1 (tamanho do "passo")
gb_model = GradientBoostingRegressor(random_state=42, n_estimators=100, learning_rate=0.1)
gb_model.fit(X_train, y_train)

y_pred = gb_model.predict(X_test)
r2_gb = r2_score(y_test, y_pred)

print(f"R² do Gradient Boosting em dados futuros: {r2_gb:.4f}")

importances_gb = pd.Series(gb_model.feature_importances_, index=X_cols).sort_values(ascending=False)
print("\nImportância das Variáveis (em %):")
print(importances_gb * 100)

MODELO: GRADIENT BOOSTING (CENÁRIO 3) - BARUERI

R² do Gradient Boosting em dados futuros: 0.7661

Importância das Variáveis (em %):
UMIDADE RELATIVA DO AR, HORARIA (%)                      41.717601
Mes                                                      22.651699
RADIACAO GLOBAL (Kj/m²)                                  14.769936
PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)    10.665828
VENTO, VELOCIDADE HORARIA (m/s)                           5.233157
Hora                                                      3.630092
VENTO, DIREÇÃO HORARIA (gr) (° (gr))                      1.331226
PRECIPITAÇÃO TOTAL, HORÁRIO (mm)                          0.000461
dtype: float64


## Teste com todas as variáveis (Sem filtro) + Hora e mês - Gradient Boosting

In [ ]:
  import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando os Dados de Barueri
nome_arquivo = 'Barueri_Sao_Paulo_2025.xlsx'
df = pd.read_excel(nome_arquivo)

# Limpeza de segurança
for col in df.columns:
    if df[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df[col] = pd.to_numeric(df[col].str.replace(',', '.'), errors='coerce')
        except:
            df[col] = pd.to_numeric(df[col], errors='coerce')

target_col = 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'

# 2. Engenharia de Atributos (Extraindo o Tempo)
df['Hora'] = df['Hora UTC'].str.replace(' UTC', '').astype(int) / 100
df['Mes'] = pd.to_datetime(df['Data']).dt.month

# 3. Variáveis Selecionadas (Todas as 11 físicas + Tempo)
X_cols = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)',
    'PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)',
    'UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)',
    'VENTO, RAJADA MAXIMA (m/s)',
    'Hora',
    'Mes'
]

df_model = df[[target_col] + X_cols].copy()

# Tratamento específico para as lacunas dos sensores de Barueri
df_model['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'] = df_model['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'].fillna(0)
df_model['RADIACAO GLOBAL (Kj/m²)'] = df_model['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_model = df_model.interpolate(method='linear', limit_direction='both').dropna()

X = df_model[X_cols]
y = df_model[target_col]

# 4. Divisão Cronológica (80% Treino / 20% Teste)
ponto_de_corte = int(len(df_model) * 0.8)
X_train, X_test = X.iloc[:ponto_de_corte], X.iloc[ponto_de_corte:]
y_train, y_test = y.iloc[:ponto_de_corte], y.iloc[ponto_de_corte:]

# =====================================================================
# MACHINE LEARNING: GRADIENT BOOSTING
# =====================================================================
print("=======================================================")
print("MODELO: GRADIENT BOOSTING (CENÁRIO 4) - BARUERI")
print("=======================================================\n")

# n_estimators=100 (100 árvores sequenciais)
# learning_rate=0.1 (tamanho do "passo")
gb_model = GradientBoostingRegressor(random_state=42, n_estimators=100, learning_rate=0.1)
gb_model.fit(X_train, y_train)

y_pred = gb_model.predict(X_test)
r2_gb = r2_score(y_test, y_pred)

print(f"R² do Gradient Boosting em dados futuros: {r2_gb:.4f}")

importances_gb = pd.Series(gb_model.feature_importances_, index=X_cols).sort_values(ascending=False)
print("\nImportância das Variáveis (em %):")
print(importances_gb * 100)

MODELO: GRADIENT BOOSTING (CENÁRIO 4) - BARUERI

R² do Gradient Boosting em dados futuros: 0.7726

Importância das Variáveis (em %):
UMIDADE RELATIVA DO AR, HORARIA (%)                      24.884077
Mes                                                      22.797960
UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)                 18.026558
RADIACAO GLOBAL (Kj/m²)                                  12.577286
PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)     6.761988
VENTO, VELOCIDADE HORARIA (m/s)                           5.150942
Hora                                                      2.896181
PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)          2.677564
VENTO, DIREÇÃO HORARIA (gr) (° (gr))                      1.407792
VENTO, RAJADA MAXIMA (m/s)                                1.250822
UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)                  1.246230
PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)           0.322600
PRECIPITAÇÃO TOTAL, HORÁRIO (mm)                          0.000

## Teste com variáveis selecionadas - XGBoost


In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.metrics import r2_score
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando os Dados de Barueri
nome_arquivo = 'Barueri_Sao_Paulo_2025.xlsx'
df = pd.read_excel(nome_arquivo)

# Limpeza de segurança
for col in df.columns:
    if df[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df[col] = pd.to_numeric(df[col].str.replace(',', '.'), errors='coerce')
        except:
            df[col] = pd.to_numeric(df[col], errors='coerce')

target_col = 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'

# 2. Variáveis Selecionadas (Cenário 1: Física pura, sem tempo)
X_cols = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)'
]

df_model = df[[target_col] + X_cols].copy()

# Tratamento de lacunas
df_model['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'] = df_model['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'].fillna(0)
df_model['RADIACAO GLOBAL (Kj/m²)'] = df_model['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_model = df_model.interpolate(method='linear', limit_direction='both').dropna()

X = df_model[X_cols]
y = df_model[target_col]

# 3. Divisão Cronológica (80% Treino / 20% Teste)
ponto_de_corte = int(len(df_model) * 0.8)
X_train, X_test = X.iloc[:ponto_de_corte], X.iloc[ponto_de_corte:]
y_train, y_test = y.iloc[:ponto_de_corte], y.iloc[ponto_de_corte:]

# =====================================================================
# MACHINE LEARNING: XGBOOST
# =====================================================================
print("=======================================================")
print("MODELO: XGBOOST (CENÁRIO 1) - BARUERI")
print("=======================================================\n")

# Usando parâmetros consistentes (100 árvores, taxa de aprendizado 0.1)
xgb_model = XGBRegressor(random_state=42, n_estimators=100, learning_rate=0.1, n_jobs=-1)
xgb_model.fit(X_train, y_train)

y_pred = xgb_model.predict(X_test)
r2_xgb = r2_score(y_test, y_pred)

print(f"R² do XGBoost em dados futuros: {r2_xgb:.4f}")

importances_xgb = pd.Series(xgb_model.feature_importances_, index=X_cols).sort_values(ascending=False)
print("\nImportância das Variáveis (em %):")
print(importances_xgb * 100)

MODELO: XGBOOST (CENÁRIO 1) - BARUERI

R² do XGBoost em dados futuros: 0.7846

Importância das Variáveis (em %):
UMIDADE RELATIVA DO AR, HORARIA (%)                      51.927040
RADIACAO GLOBAL (Kj/m²)                                  20.763836
PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)    16.399229
VENTO, VELOCIDADE HORARIA (m/s)                           7.662455
VENTO, DIREÇÃO HORARIA (gr) (° (gr))                      2.193334
PRECIPITAÇÃO TOTAL, HORÁRIO (mm)                          1.054106
dtype: float32


## Teste com todas as variáveis (Sem filtro) - XGBoost

In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.metrics import r2_score
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando os Dados de Barueri
nome_arquivo = 'Barueri_Sao_Paulo_2025.xlsx'
df = pd.read_excel(nome_arquivo)

# Limpeza de segurança
for col in df.columns:
    if df[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df[col] = pd.to_numeric(df[col].str.replace(',', '.'), errors='coerce')
        except:
            df[col] = pd.to_numeric(df[col], errors='coerce')

target_col = 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'

# 2. Variáveis Selecionadas (Cenário 2: Todas as 11 físicas, sem tempo)
X_cols = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)',
    'PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)',
    'UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)',
    'VENTO, RAJADA MAXIMA (m/s)'
]

df_model = df[[target_col] + X_cols].copy()

# Tratamento de lacunas
df_model['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'] = df_model['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'].fillna(0)
df_model['RADIACAO GLOBAL (Kj/m²)'] = df_model['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_model = df_model.interpolate(method='linear', limit_direction='both').dropna()

X = df_model[X_cols]
y = df_model[target_col]

# 3. Divisão Cronológica (80% Treino / 20% Teste)
ponto_de_corte = int(len(df_model) * 0.8)
X_train, X_test = X.iloc[:ponto_de_corte], X.iloc[ponto_de_corte:]
y_train, y_test = y.iloc[:ponto_de_corte], y.iloc[ponto_de_corte:]

# =====================================================================
# MACHINE LEARNING: XGBOOST
# =====================================================================
print("=======================================================")
print("MODELO: XGBOOST (CENÁRIO 2) - BARUERI")
print("=======================================================\n")

# n_estimators=100 (100 árvores sequenciais)
# learning_rate=0.1 (tamanho do "passo")
xgb_model = XGBRegressor(random_state=42, n_estimators=100, learning_rate=0.1, n_jobs=-1)
xgb_model.fit(X_train, y_train)

y_pred = xgb_model.predict(X_test)
r2_xgb = r2_score(y_test, y_pred)

print(f"R² do XGBoost em dados futuros: {r2_xgb:.4f}")

importances_xgb = pd.Series(xgb_model.feature_importances_, index=X_cols).sort_values(ascending=False)
print("\nImportância das Variáveis (em %):")
print(importances_xgb * 100)

MODELO: XGBOOST (CENÁRIO 2) - BARUERI

R² do XGBoost em dados futuros: 0.7953

Importância das Variáveis (em %):
UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)                 27.840212
UMIDADE RELATIVA DO AR, HORARIA (%)                      23.308552
PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)         21.328394
RADIACAO GLOBAL (Kj/m²)                                  12.106126
VENTO, VELOCIDADE HORARIA (m/s)                           4.103374
PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)     2.923579
UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)                  2.700680
VENTO, RAJADA MAXIMA (m/s)                                2.248172
PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)           1.797283
VENTO, DIREÇÃO HORARIA (gr) (° (gr))                      1.094862
PRECIPITAÇÃO TOTAL, HORÁRIO (mm)                          0.548768
dtype: float32


## Teste com variáveis selecionadas + Hora e mês - XGBoost

In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.metrics import r2_score
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando os Dados de Barueri
nome_arquivo = 'Barueri_Sao_Paulo_2025.xlsx'
df = pd.read_excel(nome_arquivo)

# Limpeza de segurança
for col in df.columns:
    if df[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df[col] = pd.to_numeric(df[col].str.replace(',', '.'), errors='coerce')
        except:
            df[col] = pd.to_numeric(df[col], errors='coerce')

target_col = 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'

# 2. Engenharia de Atributos (Extraindo o Tempo)
df['Hora'] = df['Hora UTC'].str.replace(' UTC', '').astype(int) / 100
df['Mes'] = pd.to_datetime(df['Data']).dt.month

# 3. Variáveis Selecionadas (Cenário 3: Física Limpa + Tempo)
X_cols = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)',
    'Hora',
    'Mes'
]

df_model = df[[target_col] + X_cols].copy()

# Tratamento de lacunas
df_model['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'] = df_model['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'].fillna(0)
df_model['RADIACAO GLOBAL (Kj/m²)'] = df_model['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_model = df_model.interpolate(method='linear', limit_direction='both').dropna()

X = df_model[X_cols]
y = df_model[target_col]

# 4. Divisão Cronológica (80% Treino / 20% Teste)
ponto_de_corte = int(len(df_model) * 0.8)
X_train, X_test = X.iloc[:ponto_de_corte], X.iloc[ponto_de_corte:]
y_train, y_test = y.iloc[:ponto_de_corte], y.iloc[ponto_de_corte:]

# =====================================================================
# MACHINE LEARNING: XGBOOST
# =====================================================================
print("=======================================================")
print("MODELO: XGBOOST (CENÁRIO 3) - BARUERI")
print("=======================================================\n")

# n_estimators=100 (100 árvores sequenciais)
# learning_rate=0.1 (tamanho do "passo")
xgb_model = XGBRegressor(random_state=42, n_estimators=100, learning_rate=0.1, n_jobs=-1)
xgb_model.fit(X_train, y_train)

y_pred = xgb_model.predict(X_test)
r2_xgb = r2_score(y_test, y_pred)

print(f"R² do XGBoost em dados futuros: {r2_xgb:.4f}")

importances_xgb = pd.Series(xgb_model.feature_importances_, index=X_cols).sort_values(ascending=False)
print("\nImportância das Variáveis (em %):")
print(importances_xgb * 100)

MODELO: XGBOOST (CENÁRIO 3) - BARUERI

R² do XGBoost em dados futuros: 0.7733

Importância das Variáveis (em %):
UMIDADE RELATIVA DO AR, HORARIA (%)                      36.794128
Mes                                                      31.021082
RADIACAO GLOBAL (Kj/m²)                                  18.414392
PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)     4.006721
VENTO, VELOCIDADE HORARIA (m/s)                           4.005908
Hora                                                      3.574876
VENTO, DIREÇÃO HORARIA (gr) (° (gr))                      1.457866
PRECIPITAÇÃO TOTAL, HORÁRIO (mm)                          0.725029
dtype: float32


## Teste com todas as variáveis (Sem filtro) + Hora e mês - XGBoost

In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.metrics import r2_score
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando os Dados de Barueri
nome_arquivo = 'Barueri_Sao_Paulo_2025.xlsx'
df = pd.read_excel(nome_arquivo)

# Limpeza de segurança
for col in df.columns:
    if df[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df[col] = pd.to_numeric(df[col].str.replace(',', '.'), errors='coerce')
        except:
            df[col] = pd.to_numeric(df[col], errors='coerce')

target_col = 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'

# 2. Engenharia de Atributos (Extraindo o Tempo)
df['Hora'] = df['Hora UTC'].str.replace(' UTC', '').astype(int) / 100
df['Mes'] = pd.to_datetime(df['Data']).dt.month

# 3. Variáveis Selecionadas (Cenário 4: Todas as 11 físicas + Tempo)
X_cols = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)',
    'PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)',
    'UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)',
    'VENTO, RAJADA MAXIMA (m/s)',
    'Hora',
    'Mes'
]

df_model = df[[target_col] + X_cols].copy()

# Tratamento de lacunas
df_model['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'] = df_model['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'].fillna(0)
df_model['RADIACAO GLOBAL (Kj/m²)'] = df_model['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_model = df_model.interpolate(method='linear', limit_direction='both').dropna()

X = df_model[X_cols]
y = df_model[target_col]

# 4. Divisão Cronológica (80% Treino / 20% Teste)
ponto_de_corte = int(len(df_model) * 0.8)
X_train, X_test = X.iloc[:ponto_de_corte], X.iloc[ponto_de_corte:]
y_train, y_test = y.iloc[:ponto_de_corte], y.iloc[ponto_de_corte:]

# =====================================================================
# MACHINE LEARNING: XGBOOST
# =====================================================================
print("=======================================================")
print("MODELO: XGBOOST (CENÁRIO 4) - BARUERI")
print("=======================================================\n")

# n_estimators=100 (100 árvores sequenciais)
# learning_rate=0.1 (tamanho do "passo")
xgb_model = XGBRegressor(random_state=42, n_estimators=100, learning_rate=0.1, n_jobs=-1)
xgb_model.fit(X_train, y_train)

y_pred = xgb_model.predict(X_test)
r2_xgb = r2_score(y_test, y_pred)

print(f"R² do XGBoost em dados futuros: {r2_xgb:.4f}")

importances_xgb = pd.Series(xgb_model.feature_importances_, index=X_cols).sort_values(ascending=False)
print("\nImportância das Variáveis (em %):")
print(importances_xgb * 100)

MODELO: XGBOOST (CENÁRIO 4) - BARUERI

R² do XGBoost em dados futuros: 0.7765

Importância das Variáveis (em %):
UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)                 27.062359
Mes                                                      22.262259
UMIDADE RELATIVA DO AR, HORARIA (%)                      21.363501
RADIACAO GLOBAL (Kj/m²)                                  12.822392
VENTO, VELOCIDADE HORARIA (m/s)                           2.879694
UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)                  2.459500
PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)          2.453522
PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)     2.301911
Hora                                                      2.244298
VENTO, RAJADA MAXIMA (m/s)                                1.261154
PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)           1.256193
VENTO, DIREÇÃO HORARIA (gr) (° (gr))                      0.922117
PRECIPITAÇÃO TOTAL, HORÁRIO (mm)                          0.711099
dtype: float32


## Teste com variáveis selecionadas - XGBoost tunado


In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import RandomizedSearchCV
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando e Limpando os Dados de Barueri
nome_arquivo = 'Barueri_Sao_Paulo_2025.xlsx'
df_tune1 = pd.read_excel(nome_arquivo)

for col in df_tune1.columns:
    if df_tune1[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df_tune1[col] = pd.to_numeric(df_tune1[col].str.replace(',', '.'), errors='coerce')
        except:
            df_tune1[col] = pd.to_numeric(df_tune1[col], errors='coerce')

target_col = 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'

# 2. CENÁRIO 1: Variáveis Selecionadas (Sem Tempo)
X_cols_1 = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)'
]

df_model_1 = df_tune1[[target_col] + X_cols_1].copy()

# Tratamento específico para as lacunas dos sensores de Barueri
df_model_1['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'] = df_model_1['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'].fillna(0)
df_model_1['RADIACAO GLOBAL (Kj/m²)'] = df_model_1['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_model_1 = df_model_1.interpolate(method='linear', limit_direction='both').dropna()

X_1 = df_model_1[X_cols_1]
y_1 = df_model_1[target_col]

# 3. Divisão Cronológica
ponto_1 = int(len(df_model_1) * 0.8)
X_train_1, X_test_1 = X_1.iloc[:ponto_1], X_1.iloc[ponto_1:]
y_train_1, y_test_1 = y_1.iloc[:ponto_1], y_1.iloc[ponto_1:]

print("=======================================================")
print("INICIANDO FINE TUNING: XGBOOST (CENÁRIO 1 - SEM TEMPO) - BARUERI")
print("Isso pode levar cerca de 1 a 2 minutos...")
print("=======================================================\n")

# 4. Definindo a Grade de Hiperparâmetros
param_dist = {
    'n_estimators': [100, 200, 300],          # Número de árvores
    'learning_rate': [0.01, 0.05, 0.1, 0.2],  # Passo de aprendizado
    'max_depth': [3, 4, 5, 6],                # Profundidade máxima de cada árvore
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0], # % de variáveis usadas por árvore
    'subsample': [0.7, 0.8, 0.9, 1.0]         # % de dados usados por árvore
}

# Criando o modelo base
xgb_base = XGBRegressor(random_state=42, objective='reg:squarederror')

# Criando o buscador aleatório (testará 100 combinações diferentes)
random_search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_dist,
    n_iter=100,
    scoring='r2',
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

# 5. Treinando e encontrando o melhor modelo
random_search.fit(X_train_1, y_train_1)
melhor_xgb_1 = random_search.best_estimator_

# 6. Avaliação Final
y_pred_1 = melhor_xgb_1.predict(X_test_1)
r2_1 = r2_score(y_test_1, y_pred_1)

print(f"\n--- RESULTADOS DO FINE TUNING ---")
print(f"Melhores Hiperparâmetros encontrados: {random_search.best_params_}")
print(f"R² do XGBoost Otimizado em dados futuros: {r2_1:.4f}")

importances_1 = pd.Series(melhor_xgb_1.feature_importances_, index=X_cols_1).sort_values(ascending=False)
print("\nImportância das Variáveis (em %):")
print(importances_1 * 100)

INICIANDO FINE TUNING: XGBOOST (CENÁRIO 1 - SEM TEMPO) - BARUERI
Isso pode levar cerca de 1 a 2 minutos...

Fitting 3 folds for each of 100 candidates, totalling 300 fits

--- RESULTADOS DO FINE TUNING ---
Melhores Hiperparâmetros encontrados: {'subsample': 0.9, 'n_estimators': 200, 'max_depth': 3, 'learning_rate': 0.05, 'colsample_bytree': 0.7}
R² do XGBoost Otimizado em dados futuros: 0.7821

Importância das Variáveis (em %):
RADIACAO GLOBAL (Kj/m²)                                  36.708076
UMIDADE RELATIVA DO AR, HORARIA (%)                      32.965214
PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)    21.107576
VENTO, VELOCIDADE HORARIA (m/s)                           6.656035
VENTO, DIREÇÃO HORARIA (gr) (° (gr))                      2.312884
PRECIPITAÇÃO TOTAL, HORÁRIO (mm)                          0.250212
dtype: float32


## Teste com todas as variáveis (Sem filtro) - XGBoost tunado

In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import RandomizedSearchCV
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando e Limpando os Dados de Barueri
nome_arquivo = 'Barueri_Sao_Paulo_2025.xlsx'
df_tune2 = pd.read_excel(nome_arquivo)

for col in df_tune2.columns:
    if df_tune2[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df_tune2[col] = pd.to_numeric(df_tune2[col].str.replace(',', '.'), errors='coerce')
        except:
            df_tune2[col] = pd.to_numeric(df_tune2[col], errors='coerce')

target_col = 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'

# 2. CENÁRIO 2: Todas as 11 variáveis físicas (Sem Tempo)
X_cols_2 = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)',
    'PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)',
    'UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)',
    'VENTO, RAJADA MAXIMA (m/s)'
]

df_model_2 = df_tune2[[target_col] + X_cols_2].copy()

# Tratamento específico para as lacunas dos sensores de Barueri
df_model_2['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'] = df_model_2['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'].fillna(0)
df_model_2['RADIACAO GLOBAL (Kj/m²)'] = df_model_2['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_model_2 = df_model_2.interpolate(method='linear', limit_direction='both').dropna()

X_2 = df_model_2[X_cols_2]
y_2 = df_model_2[target_col]

# 3. Divisão Cronológica
ponto_2 = int(len(df_model_2) * 0.8)
X_train_2, X_test_2 = X_2.iloc[:ponto_2], X_2.iloc[ponto_2:]
y_train_2, y_test_2 = y_2.iloc[:ponto_2], y_2.iloc[ponto_2:]

print("=======================================================")
print("INICIANDO FINE TUNING: XGBOOST (CENÁRIO 2 - 11 VARIÁVEIS)")
print("Aguarde o processamento das 100 combinações...")
print("=======================================================\n")

# 4. Definindo a Grade de Hiperparâmetros
param_dist = {
    'n_estimators': [100, 200, 300],          # Número de árvores
    'learning_rate': [0.01, 0.05, 0.1, 0.2],  # Passo de aprendizado
    'max_depth': [3, 4, 5, 6],                # Profundidade máxima de cada árvore
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0], # % de variáveis usadas por árvore
    'subsample': [0.7, 0.8, 0.9, 1.0]         # % de dados usados por árvore
}

# Criando o modelo base
xgb_base = XGBRegressor(random_state=42, objective='reg:squarederror')

# Criando o buscador aleatório
random_search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_dist,
    n_iter=100,
    scoring='r2',
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

# 5. Treinando e encontrando o melhor modelo
random_search.fit(X_train_2, y_train_2)
melhor_xgb_2 = random_search.best_estimator_

# 6. Avaliação Final
y_pred_2 = melhor_xgb_2.predict(X_test_2)
r2_2 = r2_score(y_test_2, y_pred_2)

print(f"\n--- RESULTADOS DO FINE TUNING (CENÁRIO 2) ---")
print(f"Melhores Hiperparâmetros encontrados: {random_search.best_params_}")
print(f"R² do XGBoost Otimizado em dados futuros: {r2_2:.4f}")

importances_2 = pd.Series(melhor_xgb_2.feature_importances_, index=X_cols_2).sort_values(ascending=False)
print("\nImportância das Variáveis (em %):")
print(importances_2 * 100)

INICIANDO FINE TUNING: XGBOOST (CENÁRIO 2 - 11 VARIÁVEIS)
Aguarde o processamento das 100 combinações...

Fitting 3 folds for each of 100 candidates, totalling 300 fits

--- RESULTADOS DO FINE TUNING (CENÁRIO 2) ---
Melhores Hiperparâmetros encontrados: {'subsample': 0.7, 'n_estimators': 200, 'max_depth': 4, 'learning_rate': 0.05, 'colsample_bytree': 0.9}
R² do XGBoost Otimizado em dados futuros: 0.7927

Importância das Variáveis (em %):
UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)                 26.072550
UMIDADE RELATIVA DO AR, HORARIA (%)                      20.448048
PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)         16.581621
RADIACAO GLOBAL (Kj/m²)                                  16.045034
PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)     5.141561
PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)           5.113897
VENTO, VELOCIDADE HORARIA (m/s)                           3.512402
UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)                  3.303456
VENTO, RAJADA MAXIMA (

## Teste com variáveis selecionadas + Hora e mês - XGBoost tunado

In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import RandomizedSearchCV
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando e Limpando os Dados de Barueri
nome_arquivo = 'Barueri_Sao_Paulo_2025.xlsx'
df_tune3 = pd.read_excel(nome_arquivo)

for col in df_tune3.columns:
    if df_tune3[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df_tune3[col] = pd.to_numeric(df_tune3[col].str.replace(',', '.'), errors='coerce')
        except:
            df_tune3[col] = pd.to_numeric(df_tune3[col], errors='coerce')

target_col = 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'

# 2. CENÁRIO 3: Variáveis Selecionadas + Tempo
df_tune3['Hora'] = df_tune3['Hora UTC'].str.replace(' UTC', '').astype(int) / 100
df_tune3['Mes'] = pd.to_datetime(df_tune3['Data']).dt.month

X_cols_3 = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)',
    'Hora',
    'Mes'
]

df_model_3 = df_tune3[[target_col] + X_cols_3].copy()

# Tratamento específico para as lacunas dos sensores de Barueri
df_model_3['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'] = df_model_3['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'].fillna(0)
df_model_3['RADIACAO GLOBAL (Kj/m²)'] = df_model_3['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_model_3 = df_model_3.interpolate(method='linear', limit_direction='both').dropna()

X_3 = df_model_3[X_cols_3]
y_3 = df_model_3[target_col]

# 3. Divisão Cronológica
ponto_3 = int(len(df_model_3) * 0.8)
X_train_3, X_test_3 = X_3.iloc[:ponto_3], X_3.iloc[ponto_3:]
y_train_3, y_test_3 = y_3.iloc[:ponto_3], y_3.iloc[ponto_3:]

print("=======================================================")
print("INICIANDO FINE TUNING: XGBOOST (CENÁRIO 3 - COM TEMPO) - BARUERI")
print("Aguarde o processamento das 100 combinações...")
print("=======================================================\n")

# 4. Definindo a Grade de Hiperparâmetros
param_dist = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'max_depth': [3, 4, 5, 6],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'subsample': [0.7, 0.8, 0.9, 1.0]
}

# Criando o modelo base
xgb_base = XGBRegressor(random_state=42, objective='reg:squarederror')

# Criando o buscador aleatório
random_search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_dist,
    n_iter=100,
    scoring='r2',
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

# 5. Treinando e encontrando o melhor modelo
random_search.fit(X_train_3, y_train_3)
melhor_xgb_3 = random_search.best_estimator_

# 6. Avaliação Final
y_pred_3 = melhor_xgb_3.predict(X_test_3)
r2_3 = r2_score(y_test_3, y_pred_3)

print(f"\n--- RESULTADOS DO FINE TUNING (CENÁRIO 3) ---")
print(f"Melhores Hiperparâmetros encontrados: {random_search.best_params_}")
print(f"R² do XGBoost Otimizado em dados futuros: {r2_3:.4f}")

importances_3 = pd.Series(melhor_xgb_3.feature_importances_, index=X_cols_3).sort_values(ascending=False)
print("\nImportância das Variáveis (em %):")
print(importances_3 * 100)

INICIANDO FINE TUNING: XGBOOST (CENÁRIO 3 - COM TEMPO) - BARUERI
Aguarde o processamento das 100 combinações...

Fitting 3 folds for each of 100 candidates, totalling 300 fits

--- RESULTADOS DO FINE TUNING (CENÁRIO 3) ---
Melhores Hiperparâmetros encontrados: {'subsample': 0.7, 'n_estimators': 200, 'max_depth': 5, 'learning_rate': 0.05, 'colsample_bytree': 0.7}
R² do XGBoost Otimizado em dados futuros: 0.7871

Importância das Variáveis (em %):
UMIDADE RELATIVA DO AR, HORARIA (%)                      35.021740
Mes                                                      22.183743
RADIACAO GLOBAL (Kj/m²)                                  17.254887
PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)     9.825539
Hora                                                      8.399853
VENTO, VELOCIDADE HORARIA (m/s)                           4.680459
VENTO, DIREÇÃO HORARIA (gr) (° (gr))                      1.998303
PRECIPITAÇÃO TOTAL, HORÁRIO (mm)                          0.635478
dtype: float32


## Teste com todas as variáveis (Sem filtro) + Hora e mês - XGBoost tunado

In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import RandomizedSearchCV
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando e Limpando os Dados de Barueri
nome_arquivo = 'Barueri_Sao_Paulo_2025.xlsx'
df_tune4 = pd.read_excel(nome_arquivo)

for col in df_tune4.columns:
    if df_tune4[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df_tune4[col] = pd.to_numeric(df_tune4[col].str.replace(',', '.'), errors='coerce')
        except:
            df_tune4[col] = pd.to_numeric(df_tune4[col], errors='coerce')

target_col = 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'

# 2. CENÁRIO 4: Todas as 11 variáveis físicas + Tempo
df_tune4['Hora'] = df_tune4['Hora UTC'].str.replace(' UTC', '').astype(int) / 100
df_tune4['Mes'] = pd.to_datetime(df_tune4['Data']).dt.month

X_cols_4 = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)',
    'PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)',
    'UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)',
    'VENTO, RAJADA MAXIMA (m/s)',
    'Hora',
    'Mes'
]

df_model_4 = df_tune4[[target_col] + X_cols_4].copy()

# Tratamento específico para as lacunas dos sensores de Barueri
df_model_4['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'] = df_model_4['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'].fillna(0)
df_model_4['RADIACAO GLOBAL (Kj/m²)'] = df_model_4['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_model_4 = df_model_4.interpolate(method='linear', limit_direction='both').dropna()

X_4 = df_model_4[X_cols_4]
y_4 = df_model_4[target_col]

# 3. Divisão Cronológica
ponto_4 = int(len(df_model_4) * 0.8)
X_train_4, X_test_4 = X_4.iloc[:ponto_4], X_4.iloc[ponto_4:]
y_train_4, y_test_4 = y_4.iloc[:ponto_4], y_4.iloc[ponto_4:]

print("=======================================================")
print("INICIANDO FINE TUNING: XGBOOST (CENÁRIO 4 - TUDO) - BARUERI")
print("Aguarde o processamento das 100 combinações finais...")
print("=======================================================\n")

# 4. Definindo a Grade de Hiperparâmetros
param_dist = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'max_depth': [3, 4, 5, 6],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'subsample': [0.7, 0.8, 0.9, 1.0]
}

# Criando o modelo base
xgb_base = XGBRegressor(random_state=42, objective='reg:squarederror')

# Criando o buscador aleatório
random_search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_dist,
    n_iter=100,
    scoring='r2',
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

# 5. Treinando e encontrando o melhor modelo
random_search.fit(X_train_4, y_train_4)
melhor_xgb_4 = random_search.best_estimator_

# 6. Avaliação Final
y_pred_4 = melhor_xgb_4.predict(X_test_4)
r2_4 = r2_score(y_test_4, y_pred_4)

print(f"\n--- RESULTADOS DO FINE TUNING (CENÁRIO 4) ---")
print(f"Melhores Hiperparâmetros encontrados: {random_search.best_params_}")
print(f"R² do XGBoost Otimizado em dados futuros: {r2_4:.4f}")

importances_4 = pd.Series(melhor_xgb_4.feature_importances_, index=X_cols_4).sort_values(ascending=False)
print("\nImportância das Variáveis (em %):")
print(importances_4 * 100)
print("=======================================================")

INICIANDO FINE TUNING: XGBOOST (CENÁRIO 4 - TUDO) - BARUERI
Aguarde o processamento das 100 combinações finais...

Fitting 3 folds for each of 100 candidates, totalling 300 fits

--- RESULTADOS DO FINE TUNING (CENÁRIO 4) ---
Melhores Hiperparâmetros encontrados: {'subsample': 0.7, 'n_estimators': 100, 'max_depth': 3, 'learning_rate': 0.2, 'colsample_bytree': 0.9}
R² do XGBoost Otimizado em dados futuros: 0.7806

Importância das Variáveis (em %):
UMIDADE RELATIVA DO AR, HORARIA (%)                      27.351833
RADIACAO GLOBAL (Kj/m²)                                  20.189732
Mes                                                      16.926273
UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)                 12.196507
PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)     6.908581
PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)           4.946645
UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)                  3.694994
Hora                                                      2.253241
VENTO, VELOCID

## Teste com variáveis selecionadas - PCA

In [ ]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando os Dados de Barueri
nome_arquivo = 'Barueri_Sao_Paulo_2025.xlsx'
df = pd.read_excel(nome_arquivo)

# Limpeza e conversão de segurança
for col in df.columns:
    if df[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df[col] = pd.to_numeric(df[col].str.replace(',', '.'), errors='coerce')
        except:
            df[col] = pd.to_numeric(df[col], errors='coerce')

# 2. Definição das variáveis do Cenário 1
X_cols_1 = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)'
]

# Isolando e tratando nulos
df_pca1 = df[X_cols_1].copy()
df_pca1['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'] = df_pca1['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'].fillna(0)
df_pca1['RADIACAO GLOBAL (Kj/m²)'] = df_pca1['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_pca1 = df_pca1.interpolate(method='linear', limit_direction='both').dropna()

# 3. Padronização Obrigatória (Z-score)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_pca1)

# 4. Executando o PCA
pca = PCA(random_state=42)
pca.fit(X_scaled)

# 5. Calculando a Variância Explicada
var_explicada = pca.explained_variance_ratio_ * 100

print("=======================================================")
print("PCA: BARUERI - CENÁRIO 1 (SEM TEMPO)")
print("=======================================================\n")
print(f"Variância no Componente 1 (PC1): {var_explicada[0]:.2f}%")
print(f"Variância no Componente 2 (PC2): {var_explicada[1]:.2f}%")
print(f"Variância Acumulada (PC1 + PC2): {var_explicada[0] + var_explicada[1]:.2f}%\n")

# 6. Criando a Tabela de Cargas (Loadings) para entender o impacto
loadings = pd.DataFrame(
    pca.components_.T,
    columns=[f'PC{i+1}' for i in range(len(X_cols_1))],
    index=X_cols_1
)

# Adiciona coluna de impacto absoluto para ordenar pelo PC1 (o eixo mais importante)
loadings['Impacto_Absoluto_PC1'] = loadings['PC1'].abs()
tabela_final = loadings.sort_values(by='Impacto_Absoluto_PC1', ascending=False)

print("Matriz de Cargas no PC1 (Ordenado por Relevância Estrutural):")
print(tabela_final[['PC1', 'Impacto_Absoluto_PC1']])

PCA: BARUERI - CENÁRIO 1 (SEM TEMPO)

Variância no Componente 1 (PC1): 29.75%
Variância no Componente 2 (PC2): 20.69%
Variância Acumulada (PC1 + PC2): 50.44%

Matriz de Cargas no PC1 (Ordenado por Relevância Estrutural):
                                                         PC1  \
RADIACAO GLOBAL (Kj/m²)                             0.632250   
UMIDADE RELATIVA DO AR, HORARIA (%)                -0.625480   
VENTO, VELOCIDADE HORARIA (m/s)                     0.282925   
PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARI... -0.280594   
VENTO, DIREÇÃO HORARIA (gr) (° (gr))                0.223998   
PRECIPITAÇÃO TOTAL, HORÁRIO (mm)                   -0.008909   

                                                    Impacto_Absoluto_PC1  
RADIACAO GLOBAL (Kj/m²)                                         0.632250  
UMIDADE RELATIVA DO AR, HORARIA (%)                             0.625480  
VENTO, VELOCIDADE HORARIA (m/s)                                 0.282925  
PRESSAO ATMOSFERICA AO NIVEL D

## Teste com todas as variáveis (Sem filtro) - PCA

In [ ]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando os Dados de Barueri
nome_arquivo = 'Barueri_Sao_Paulo_2025.xlsx'
df = pd.read_excel(nome_arquivo)

# Limpeza e conversão de segurança
for col in df.columns:
    if df[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df[col] = pd.to_numeric(df[col].str.replace(',', '.'), errors='coerce')
        except:
            df[col] = pd.to_numeric(df[col], errors='coerce')

# 2. Definição das variáveis do Cenário 2 (11 variáveis)
X_cols_2 = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)',
    'PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)',
    'UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)',
    'VENTO, RAJADA MAXIMA (m/s)'
]

# Isolando e tratando nulos
df_pca2 = df[X_cols_2].copy()
df_pca2['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'] = df_pca2['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'].fillna(0)
df_pca2['RADIACAO GLOBAL (Kj/m²)'] = df_pca2['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_pca2 = df_pca2.interpolate(method='linear', limit_direction='both').dropna()

# 3. Padronização Obrigatória (Z-score)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_pca2)

# 4. Executando o PCA
pca = PCA(random_state=42)
pca.fit(X_scaled)

# 5. Calculando a Variância Explicada
var_explicada = pca.explained_variance_ratio_ * 100

print("=======================================================")
print("PCA: BARUERI - CENÁRIO 2 (11 VARIÁVEIS, SEM TEMPO)")
print("=======================================================\n")
print(f"Variância no Componente 1 (PC1): {var_explicada[0]:.2f}%")
print(f"Variância no Componente 2 (PC2): {var_explicada[1]:.2f}%")
print(f"Variância Acumulada (PC1 + PC2): {var_explicada[0] + var_explicada[1]:.2f}%\n")

# 6. Criando a Tabela de Cargas (Loadings) para entender o impacto
loadings = pd.DataFrame(
    pca.components_.T,
    columns=[f'PC{i+1}' for i in range(len(X_cols_2))],
    index=X_cols_2
)

# Adiciona coluna de impacto absoluto para ordenar pelo PC1
loadings['Impacto_Absoluto_PC1'] = loadings['PC1'].abs()
tabela_final = loadings.sort_values(by='Impacto_Absoluto_PC1', ascending=False)

print("Matriz de Cargas no PC1 (Ordenado por Relevância Estrutural):")
print(tabela_final[['PC1', 'Impacto_Absoluto_PC1']])

PCA: BARUERI - CENÁRIO 2 (11 VARIÁVEIS, SEM TEMPO)

Variância no Componente 1 (PC1): 35.58%
Variância no Componente 2 (PC2): 26.78%
Variância Acumulada (PC1 + PC2): 62.36%

Matriz de Cargas no PC1 (Ordenado por Relevância Estrutural):
                                                         PC1  \
UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)            0.413180   
UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)            0.406878   
UMIDADE RELATIVA DO AR, HORARIA (%)                 0.406519   
RADIACAO GLOBAL (Kj/m²)                            -0.346137   
VENTO, RAJADA MAXIMA (m/s)                         -0.292198   
PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARI...  0.292142   
PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)    0.285548   
PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)     0.281773   
VENTO, VELOCIDADE HORARIA (m/s)                    -0.173400   
VENTO, DIREÇÃO HORARIA (gr) (° (gr))               -0.129025   
PRECIPITAÇÃO TOTAL, HORÁRIO (mm)                   -0.016961 

## Teste com variáveis selecionadas + Hora e mês - PCA

In [ ]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando os Dados de Barueri
nome_arquivo = 'Barueri_Sao_Paulo_2025.xlsx'
df = pd.read_excel(nome_arquivo)

# Limpeza e conversão de segurança
for col in df.columns:
    if df[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df[col] = pd.to_numeric(df[col].str.replace(',', '.'), errors='coerce')
        except:
            df[col] = pd.to_numeric(df[col], errors='coerce')

# 2. Definição das variáveis do Cenário 3 (Física Limpa + Tempo)
df['Hora'] = df['Hora UTC'].str.replace(' UTC', '').astype(int) / 100
df['Mes'] = pd.to_datetime(df['Data']).dt.month

X_cols_3 = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)',
    'Hora',
    'Mes'
]

# Isolando e tratando nulos
df_pca3 = df[X_cols_3].copy()
df_pca3['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'] = df_pca3['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'].fillna(0)
df_pca3['RADIACAO GLOBAL (Kj/m²)'] = df_pca3['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_pca3 = df_pca3.interpolate(method='linear', limit_direction='both').dropna()

# 3. Padronização Obrigatória (Z-score)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_pca3)

# 4. Executando o PCA
pca = PCA(random_state=42)
pca.fit(X_scaled)

# 5. Calculando a Variância Explicada
var_explicada = pca.explained_variance_ratio_ * 100

print("=======================================================")
print("PCA: BARUERI - CENÁRIO 3 (FÍSICA + TEMPO)")
print("=======================================================\n")
print(f"Variância no Componente 1 (PC1): {var_explicada[0]:.2f}%")
print(f"Variância no Componente 2 (PC2): {var_explicada[1]:.2f}%")
print(f"Variância Acumulada (PC1 + PC2): {var_explicada[0] + var_explicada[1]:.2f}%\n")

# 6. Criando a Tabela de Cargas (Loadings) para entender o impacto
loadings = pd.DataFrame(
    pca.components_.T,
    columns=[f'PC{i+1}' for i in range(len(X_cols_3))],
    index=X_cols_3
)

# Adiciona coluna de impacto absoluto para ordenar pelo PC1
loadings['Impacto_Absoluto_PC1'] = loadings['PC1'].abs()
tabela_final = loadings.sort_values(by='Impacto_Absoluto_PC1', ascending=False)

print("Matriz de Cargas no PC1 (Ordenado por Relevância Estrutural):")
print(tabela_final[['PC1', 'Impacto_Absoluto_PC1']])

PCA: BARUERI - CENÁRIO 3 (FÍSICA + TEMPO)

Variância no Componente 1 (PC1): 26.50%
Variância no Componente 2 (PC2): 18.79%
Variância Acumulada (PC1 + PC2): 45.29%

Matriz de Cargas no PC1 (Ordenado por Relevância Estrutural):
                                                         PC1  \
UMIDADE RELATIVA DO AR, HORARIA (%)                 0.576247   
RADIACAO GLOBAL (Kj/m²)                            -0.516216   
Hora                                               -0.452777   
VENTO, VELOCIDADE HORARIA (m/s)                    -0.325110   
PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARI...  0.190227   
Mes                                                -0.178773   
VENTO, DIREÇÃO HORARIA (gr) (° (gr))               -0.150169   
PRECIPITAÇÃO TOTAL, HORÁRIO (mm)                   -0.007805   

                                                    Impacto_Absoluto_PC1  
UMIDADE RELATIVA DO AR, HORARIA (%)                             0.576247  
RADIACAO GLOBAL (Kj/m²)                        

## Teste com todas as variáveis (Sem filtro) + Hora e mês - PCA

In [ ]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando os Dados de Barueri
nome_arquivo = 'Barueri_Sao_Paulo_2025.xlsx'
df = pd.read_excel(nome_arquivo)

# Limpeza e conversão de segurança
for col in df.columns:
    if df[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df[col] = pd.to_numeric(df[col].str.replace(',', '.'), errors='coerce')
        except:
            df[col] = pd.to_numeric(df[col], errors='coerce')

# 2. Definição das variáveis do Cenário 4 (Todas as 11 Físicas + Tempo)
df['Hora'] = df['Hora UTC'].str.replace(' UTC', '').astype(int) / 100
df['Mes'] = pd.to_datetime(df['Data']).dt.month

X_cols_4 = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)',
    'PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)',
    'UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)',
    'VENTO, RAJADA MAXIMA (m/s)',
    'Hora',
    'Mes'
]

# Isolando e tratando nulos
df_pca4 = df[X_cols_4].copy()
df_pca4['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'] = df_pca4['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'].fillna(0)
df_pca4['RADIACAO GLOBAL (Kj/m²)'] = df_pca4['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_pca4 = df_pca4.interpolate(method='linear', limit_direction='both').dropna()

# 3. Padronização Obrigatória (Z-score)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_pca4)

# 4. Executando o PCA
pca = PCA(random_state=42)
pca.fit(X_scaled)

# 5. Calculando a Variância Explicada
var_explicada = pca.explained_variance_ratio_ * 100

print("=======================================================")
print("PCA: BARUERI - CENÁRIO 4 (11 FÍSICAS + TEMPO)")
print("=======================================================\n")
print(f"Variância no Componente 1 (PC1): {var_explicada[0]:.2f}%")
print(f"Variância no Componente 2 (PC2): {var_explicada[1]:.2f}%")
print(f"Variância Acumulada (PC1 + PC2): {var_explicada[0] + var_explicada[1]:.2f}%\n")

# 6. Criando a Tabela de Cargas (Loadings) para entender o impacto
loadings = pd.DataFrame(
    pca.components_.T,
    columns=[f'PC{i+1}' for i in range(len(X_cols_4))],
    index=X_cols_4
)

# Adiciona coluna de impacto absoluto para ordenar pelo PC1
loadings['Impacto_Absoluto_PC1'] = loadings['PC1'].abs()
tabela_final = loadings.sort_values(by='Impacto_Absoluto_PC1', ascending=False)

print("Matriz de Cargas no PC1 (Ordenado por Relevância Estrutural):")
print(tabela_final[['PC1', 'Impacto_Absoluto_PC1']])

PCA: BARUERI - CENÁRIO 4 (11 FÍSICAS + TEMPO)

Variância no Componente 1 (PC1): 32.31%
Variância no Componente 2 (PC2): 23.05%
Variância Acumulada (PC1 + PC2): 55.36%

Matriz de Cargas no PC1 (Ordenado por Relevância Estrutural):
                                                         PC1  \
UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)            0.419740   
UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)            0.411932   
UMIDADE RELATIVA DO AR, HORARIA (%)                 0.411781   
RADIACAO GLOBAL (Kj/m²)                            -0.333495   
VENTO, RAJADA MAXIMA (m/s)                         -0.286017   
Hora                                               -0.277881   
PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARI...  0.236248   
PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)    0.230629   
PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)     0.226304   
VENTO, VELOCIDADE HORARIA (m/s)                    -0.186002   
VENTO, DIREÇÃO HORARIA (gr) (° (gr))               -0.109696   
Me

## Teste com variáveis selecionadas - XGBoost (Cross-Validation)

In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.model_selection import cross_val_score, KFold, TimeSeriesSplit
from sklearn.metrics import r2_score
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando os Dados de Barueri
nome_arquivo = 'Barueri_Sao_Paulo_2025.xlsx'
df = pd.read_excel(nome_arquivo)

# Limpeza padrão de segurança para garantir dados numéricos
for col in df.columns:
    if df[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df[col] = pd.to_numeric(df[col].str.replace(',', '.'), errors='coerce')
        except:
            df[col] = pd.to_numeric(df[col], errors='coerce')

target_col = 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'

# 2. Variáveis do Cenário 1 (Física pura, sem variáveis de tempo)
X_cols = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)'
]

df_model = df[[target_col] + X_cols].copy()

# Tratamento específico para as lacunas dos sensores de Barueri
df_model['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'] = df_model['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'].fillna(0)
df_model['RADIACAO GLOBAL (Kj/m²)'] = df_model['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_model = df_model.interpolate(method='linear', limit_direction='both').dropna()

X = df_model[X_cols]
y = df_model[target_col]

# 3. Configurando o modelo com os Hiperparâmetros Otimizados do seu Tuning (XGBoost C1)
modelo_tunado = XGBRegressor(
    subsample=0.9,
    n_estimators=200,
    max_depth=3,
    learning_rate=0.05,
    colsample_bytree=0.7,
    random_state=42,
    objective='reg:squarederror',
    n_jobs=-1
)

# =====================================================================
# ABORDAGEM A: K-FOLD TRADICIONAL (VALIDAÇÃO ESTRUTURAL / SHUFFLE TRUE)
# =====================================================================
print("=====================================================================")
print("MÉTODO A: K-FOLD TRADICIONAL - BARUERI (CENÁRIO 1)")
print("Foco: Estabilidade Estrutural das Regras Físicas")
print("=====================================================================")

cv_kfold = KFold(n_splits=5, shuffle=True, random_state=42)
scores_kfold = cross_val_score(modelo_tunado, X, y, cv=cv_kfold, scoring='r2', n_jobs=-1)

for i, score in enumerate(scores_kfold):
    print(f"Dobra Aleatória {i+1}: R² = {score:.4f}")

print("\n--- RESUMO MÉTODO A (K-FOLD) ---")
print(f"R² Médio Global: {np.mean(scores_kfold):.4f}")
print(f"Desvio Padrão das Dobras: {np.std(scores_kfold):.4f}\n")


# =====================================================================
# ABORDAGEM B: TIME SERIES SPLIT (VALIDAÇÃO TEMPORAL REAL / SEM SHUFFLE)
# =====================================================================
print("=====================================================================")
print("MÉTODO B: TIME SERIES SPLIT - BARUERI (CENÁRIO 1)")
print("Foco: Capacidade Real de Previsão Sem Espiar o Futuro")
print("=====================================================================")

tscv = TimeSeriesSplit(n_splits=5)
scores_temporal = []

for fold, (train_index, test_index) in enumerate(tscv.split(X)):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    modelo_tunado.fit(X_train, y_train)
    score = modelo_tunado.score(X_test, y_test)
    scores_temporal.append(score)
    print(f"Dobra Temporal {fold+1} (Treino: {len(X_train)}h | Teste: {len(X_test)}h): R² = {score:.4f}")

print("\n--- RESUMO MÉTODO B (TIME SERIES SPLIT) ---")
print(f"R² Médio Global Real: {np.mean(scores_temporal):.4f}")
print(f"Desvio Padrão Temporal: {np.std(scores_temporal):.4f}")
print("=====================================================================")

MÉTODO A: K-FOLD TRADICIONAL - BARUERI (CENÁRIO 1)
Foco: Estabilidade Estrutural das Regras Físicas
Dobra Aleatória 1: R² = 0.8500
Dobra Aleatória 2: R² = 0.8408
Dobra Aleatória 3: R² = 0.8406
Dobra Aleatória 4: R² = 0.8456
Dobra Aleatória 5: R² = 0.8485

--- RESUMO MÉTODO A (K-FOLD) ---
R² Médio Global: 0.8451
Desvio Padrão das Dobras: 0.0039

MÉTODO B: TIME SERIES SPLIT - BARUERI (CENÁRIO 1)
Foco: Capacidade Real de Previsão Sem Espiar o Futuro
Dobra Temporal 1 (Treino: 1460h | Teste: 1460h): R² = 0.3130
Dobra Temporal 2 (Treino: 2920h | Teste: 1460h): R² = -0.0979
Dobra Temporal 3 (Treino: 4380h | Teste: 1460h): R² = 0.3648
Dobra Temporal 4 (Treino: 5840h | Teste: 1460h): R² = 0.7105
Dobra Temporal 5 (Treino: 7300h | Teste: 1460h): R² = 0.7721

--- RESUMO MÉTODO B (TIME SERIES SPLIT) ---
R² Médio Global Real: 0.4125
Desvio Padrão Temporal: 0.3133


## Teste com todas as variáveis (Sem filtro) - XGBoost (Cross-Validation)

In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.model_selection import cross_val_score, KFold, TimeSeriesSplit
from sklearn.metrics import r2_score
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando os Dados de Barueri
nome_arquivo = 'Barueri_Sao_Paulo_2025.xlsx'
df = pd.read_excel(nome_arquivo)

# Limpeza padrão
for col in df.columns:
    if df[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df[col] = pd.to_numeric(df[col].str.replace(',', '.'), errors='coerce')
        except:
            df[col] = pd.to_numeric(df[col], errors='coerce')

target_col = 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'

# 2. Variáveis do Cenário 2 (11 Variáveis físicas, sem tempo)
X_cols = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)',
    'PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)',
    'UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)',
    'VENTO, RAJADA MAXIMA (m/s)'
]

df_model = df[[target_col] + X_cols].copy()

# Tratamento de lacunas
df_model['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'] = df_model['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'].fillna(0)
df_model['RADIACAO GLOBAL (Kj/m²)'] = df_model['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_model = df_model.interpolate(method='linear', limit_direction='both').dropna()

X = df_model[X_cols]
y = df_model[target_col]

# 3. Configurando o modelo com Hiperparâmetros Otimizados do Cenário 2
modelo_tunado = XGBRegressor(
    subsample=0.7,
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    colsample_bytree=0.9,
    random_state=42,
    objective='reg:squarederror',
    n_jobs=-1
)

# =====================================================================
# ABORDAGEM A: K-FOLD TRADICIONAL (VALIDAÇÃO ESTRUTURAL / SHUFFLE TRUE)
# =====================================================================
print("=====================================================================")
print("MÉTODO A: K-FOLD TRADICIONAL - BARUERI (CENÁRIO 2)")
print("Foco: Estabilidade Estrutural das Regras Físicas")
print("=====================================================================")

cv_kfold = KFold(n_splits=5, shuffle=True, random_state=42)
scores_kfold = cross_val_score(modelo_tunado, X, y, cv=cv_kfold, scoring='r2', n_jobs=-1)

for i, score in enumerate(scores_kfold):
    print(f"Dobra Aleatória {i+1}: R² = {score:.4f}")

print("\n--- RESUMO MÉTODO A (K-FOLD) ---")
print(f"R² Médio Global: {np.mean(scores_kfold):.4f}")
print(f"Desvio Padrão das Dobras: {np.std(scores_kfold):.4f}\n")


# =====================================================================
# ABORDAGEM B: TIME SERIES SPLIT (VALIDAÇÃO TEMPORAL REAL / SEM SHUFFLE)
# =====================================================================
print("=====================================================================")
print("MÉTODO B: TIME SERIES SPLIT - BARUERI (CENÁRIO 2)")
print("Foco: Capacidade Real de Previsão Sem Espiar o Futuro")
print("=====================================================================")

tscv = TimeSeriesSplit(n_splits=5)
scores_temporal = []

for fold, (train_index, test_index) in enumerate(tscv.split(X)):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    modelo_tunado.fit(X_train, y_train)
    score = modelo_tunado.score(X_test, y_test)
    scores_temporal.append(score)
    print(f"Dobra Temporal {fold+1} (Treino: {len(X_train)}h | Teste: {len(X_test)}h): R² = {score:.4f}")

print("\n--- RESUMO MÉTODO B (TIME SERIES SPLIT) ---")
print(f"R² Médio Global Real: {np.mean(scores_temporal):.4f}")
print(f"Desvio Padrão Temporal: {np.std(scores_temporal):.4f}")
print("=====================================================================")

MÉTODO A: K-FOLD TRADICIONAL - BARUERI (CENÁRIO 2)
Foco: Estabilidade Estrutural das Regras Físicas
Dobra Aleatória 1: R² = 0.8752
Dobra Aleatória 2: R² = 0.8738
Dobra Aleatória 3: R² = 0.8723
Dobra Aleatória 4: R² = 0.8741
Dobra Aleatória 5: R² = 0.8819

--- RESUMO MÉTODO A (K-FOLD) ---
R² Médio Global: 0.8755
Desvio Padrão das Dobras: 0.0034

MÉTODO B: TIME SERIES SPLIT - BARUERI (CENÁRIO 2)
Foco: Capacidade Real de Previsão Sem Espiar o Futuro
Dobra Temporal 1 (Treino: 1460h | Teste: 1460h): R² = 0.2854
Dobra Temporal 2 (Treino: 2920h | Teste: 1460h): R² = -0.2754
Dobra Temporal 3 (Treino: 4380h | Teste: 1460h): R² = 0.3640
Dobra Temporal 4 (Treino: 5840h | Teste: 1460h): R² = 0.6539
Dobra Temporal 5 (Treino: 7300h | Teste: 1460h): R² = 0.7881

--- RESUMO MÉTODO B (TIME SERIES SPLIT) ---
R² Médio Global Real: 0.3632
Desvio Padrão Temporal: 0.3685


## Teste com variáveis selecionadas + Hora e mês - XGBoost (Cross-Validation)

In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.model_selection import cross_val_score, KFold, TimeSeriesSplit
from sklearn.metrics import r2_score
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando os Dados de Barueri
nome_arquivo = 'Barueri_Sao_Paulo_2025.xlsx'
df = pd.read_excel(nome_arquivo)

# Limpeza padrão
for col in df.columns:
    if df[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df[col] = pd.to_numeric(df[col].str.replace(',', '.'), errors='coerce')
        except:
            df[col] = pd.to_numeric(df[col], errors='coerce')

target_col = 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'

# 2. Variáveis do Cenário 3 (Física + Tempo)
df['Hora'] = df['Hora UTC'].str.replace(' UTC', '').astype(int) / 100
df['Mes'] = pd.to_datetime(df['Data']).dt.month

X_cols = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)',
    'Hora',
    'Mes'
]

df_model = df[[target_col] + X_cols].copy()

# Tratamento de lacunas
df_model['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'] = df_model['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'].fillna(0)
df_model['RADIACAO GLOBAL (Kj/m²)'] = df_model['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_model = df_model.interpolate(method='linear', limit_direction='both').dropna()

X = df_model[X_cols]
y = df_model[target_col]

# 3. Configurando o modelo com Hiperparâmetros Otimizados do Cenário 3
modelo_tunado = XGBRegressor(
    subsample=0.7,
    n_estimators=200,
    max_depth=5,
    learning_rate=0.05,
    colsample_bytree=0.7,
    random_state=42,
    objective='reg:squarederror',
    n_jobs=-1
)

# =====================================================================
# ABORDAGEM A: K-FOLD TRADICIONAL (VALIDAÇÃO ESTRUTURAL / SHUFFLE TRUE)
# =====================================================================
print("=====================================================================")
print("MÉTODO A: K-FOLD TRADICIONAL - BARUERI (CENÁRIO 3)")
print("Foco: Estabilidade Estrutural das Regras Físicas")
print("=====================================================================")

cv_kfold = KFold(n_splits=5, shuffle=True, random_state=42)
scores_kfold = cross_val_score(modelo_tunado, X, y, cv=cv_kfold, scoring='r2', n_jobs=-1)

for i, score in enumerate(scores_kfold):
    print(f"Dobra Aleatória {i+1}: R² = {score:.4f}")

print("\n--- RESUMO MÉTODO A (K-FOLD) ---")
print(f"R² Médio Global: {np.mean(scores_kfold):.4f}")
print(f"Desvio Padrão das Dobras: {np.std(scores_kfold):.4f}\n")


# =====================================================================
# ABORDAGEM B: TIME SERIES SPLIT (VALIDAÇÃO TEMPORAL REAL / SEM SHUFFLE)
# =====================================================================
print("=====================================================================")
print("MÉTODO B: TIME SERIES SPLIT - BARUERI (CENÁRIO 3)")
print("Foco: Capacidade Real de Previsão Sem Espiar o Futuro")
print("=====================================================================")

tscv = TimeSeriesSplit(n_splits=5)
scores_temporal = []

for fold, (train_index, test_index) in enumerate(tscv.split(X)):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    modelo_tunado.fit(X_train, y_train)
    score = modelo_tunado.score(X_test, y_test)
    scores_temporal.append(score)
    print(f"Dobra Temporal {fold+1} (Treino: {len(X_train)}h | Teste: {len(X_test)}h): R² = {score:.4f}")

print("\n--- RESUMO MÉTODO B (TIME SERIES SPLIT) ---")
print(f"R² Médio Global Real: {np.mean(scores_temporal):.4f}")
print(f"Desvio Padrão Temporal: {np.std(scores_temporal):.4f}")
print("=====================================================================")

MÉTODO A: K-FOLD TRADICIONAL - BARUERI (CENÁRIO 3)
Foco: Estabilidade Estrutural das Regras Físicas
Dobra Aleatória 1: R² = 0.9307
Dobra Aleatória 2: R² = 0.9252
Dobra Aleatória 3: R² = 0.9207
Dobra Aleatória 4: R² = 0.9252
Dobra Aleatória 5: R² = 0.9261

--- RESUMO MÉTODO A (K-FOLD) ---
R² Médio Global: 0.9256
Desvio Padrão das Dobras: 0.0032

MÉTODO B: TIME SERIES SPLIT - BARUERI (CENÁRIO 3)
Foco: Capacidade Real de Previsão Sem Espiar o Futuro
Dobra Temporal 1 (Treino: 1460h | Teste: 1460h): R² = 0.1729
Dobra Temporal 2 (Treino: 2920h | Teste: 1460h): R² = 0.4589
Dobra Temporal 3 (Treino: 4380h | Teste: 1460h): R² = 0.6312
Dobra Temporal 4 (Treino: 5840h | Teste: 1460h): R² = 0.7092
Dobra Temporal 5 (Treino: 7300h | Teste: 1460h): R² = 0.7852

--- RESUMO MÉTODO B (TIME SERIES SPLIT) ---
R² Médio Global Real: 0.5515
Desvio Padrão Temporal: 0.2181


## Teste com todas as variáveis (Sem filtro) + Hora e mês - XGBoost (Cross-Validation)

In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.model_selection import cross_val_score, KFold, TimeSeriesSplit
from sklearn.metrics import r2_score
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando os Dados de Barueri
nome_arquivo = 'Barueri_Sao_Paulo_2025.xlsx'
df = pd.read_excel(nome_arquivo)

# Limpeza padrão
for col in df.columns:
    if df[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df[col] = pd.to_numeric(df[col].str.replace(',', '.'), errors='coerce')
        except:
            df[col] = pd.to_numeric(df[col], errors='coerce')

target_col = 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'

# 2. Variáveis do Cenário 4 (Todas as 11 Físicas + Tempo)
df['Hora'] = df['Hora UTC'].str.replace(' UTC', '').astype(int) / 100
df['Mes'] = pd.to_datetime(df['Data']).dt.month

X_cols = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)',
    'PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)',
    'UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)',
    'VENTO, RAJADA MAXIMA (m/s)',
    'Hora',
    'Mes'
]

df_model = df[[target_col] + X_cols].copy()

# Tratamento de lacunas
df_model['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'] = df_model['PRECIPITAÇÃO TOTAL, HORÁRIO (mm)'].fillna(0)
df_model['RADIACAO GLOBAL (Kj/m²)'] = df_model['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_model = df_model.interpolate(method='linear', limit_direction='both').dropna()

X = df_model[X_cols]
y = df_model[target_col]

# 3. Configurando o modelo com Hiperparâmetros Otimizados do Cenário 4
modelo_tunado = XGBRegressor(
    subsample=0.7,
    n_estimators=100,
    max_depth=3,
    learning_rate=0.2,
    colsample_bytree=0.9,
    random_state=42,
    objective='reg:squarederror',
    n_jobs=-1
)

# =====================================================================
# ABORDAGEM A: K-FOLD TRADICIONAL (VALIDAÇÃO ESTRUTURAL / SHUFFLE TRUE)
# =====================================================================
print("=====================================================================")
print("MÉTODO A: K-FOLD TRADICIONAL - BARUERI (CENÁRIO 4)")
print("Foco: Estabilidade Estrutural das Regras Físicas")
print("=====================================================================")

cv_kfold = KFold(n_splits=5, shuffle=True, random_state=42)
scores_kfold = cross_val_score(modelo_tunado, X, y, cv=cv_kfold, scoring='r2', n_jobs=-1)

for i, score in enumerate(scores_kfold):
    print(f"Dobra Aleatória {i+1}: R² = {score:.4f}")

print("\n--- RESUMO MÉTODO A (K-FOLD) ---")
print(f"R² Médio Global: {np.mean(scores_kfold):.4f}")
print(f"Desvio Padrão das Dobras: {np.std(scores_kfold):.4f}\n")


# =====================================================================
# ABORDAGEM B: TIME SERIES SPLIT (VALIDAÇÃO TEMPORAL REAL / SEM SHUFFLE)
# =====================================================================
print("=====================================================================")
print("MÉTODO B: TIME SERIES SPLIT - BARUERI (CENÁRIO 4)")
print("Foco: Capacidade Real de Previsão Sem Espiar o Futuro")
print("=====================================================================")

tscv = TimeSeriesSplit(n_splits=5)
scores_temporal = []

for fold, (train_index, test_index) in enumerate(tscv.split(X)):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    modelo_tunado.fit(X_train, y_train)
    score = modelo_tunado.score(X_test, y_test)
    scores_temporal.append(score)
    print(f"Dobra Temporal {fold+1} (Treino: {len(X_train)}h | Teste: {len(X_test)}h): R² = {score:.4f}")

print("\n--- RESUMO MÉTODO B (TIME SERIES SPLIT) ---")
print(f"R² Médio Global Real: {np.mean(scores_temporal):.4f}")
print(f"Desvio Padrão Temporal: {np.std(scores_temporal):.4f}")
print("=====================================================================")

MÉTODO A: K-FOLD TRADICIONAL - BARUERI (CENÁRIO 4)
Foco: Estabilidade Estrutural das Regras Físicas
Dobra Aleatória 1: R² = 0.9189
Dobra Aleatória 2: R² = 0.9166
Dobra Aleatória 3: R² = 0.9077
Dobra Aleatória 4: R² = 0.9132
Dobra Aleatória 5: R² = 0.9101

--- RESUMO MÉTODO A (K-FOLD) ---
R² Médio Global: 0.9133
Desvio Padrão das Dobras: 0.0041

MÉTODO B: TIME SERIES SPLIT - BARUERI (CENÁRIO 4)
Foco: Capacidade Real de Previsão Sem Espiar o Futuro
Dobra Temporal 1 (Treino: 1460h | Teste: 1460h): R² = 0.2544
Dobra Temporal 2 (Treino: 2920h | Teste: 1460h): R² = 0.5356
Dobra Temporal 3 (Treino: 4380h | Teste: 1460h): R² = 0.4953
Dobra Temporal 4 (Treino: 5840h | Teste: 1460h): R² = 0.6978
Dobra Temporal 5 (Treino: 7300h | Teste: 1460h): R² = 0.7823

--- RESUMO MÉTODO B (TIME SERIES SPLIT) ---
R² Médio Global Real: 0.5531
Desvio Padrão Temporal: 0.1824
